#  Nuclear spin control

To handle the nuclear spin control issue, there are so far 3 perspecrtives:
1. Focus on single spin and focus on the conditional rotation $n_0\cdot n_1$; as long as the spectral width is finer under UDD, then it is fine. The idel spins are allowed to have non-identity evolutions, as long as being disentangled. [This is the philosophy of Hanson PRL 2012. I guess they will treat the evoluiton of idle qubits -notice they are NOT entangled- as "fixable" (by tracking theri accumulation and counteract them?).]

2. Focus on the evolution of the idle spins (notice even their HFI =0, their evolution is not identity - there are drifting evolution due to Larmor precission). The cost function might be based on $U_{ideal}$ which is entangling evolution on target and identity on idle spins.

3. Open quantum system formalism: Trace out the idle spins and focus on the evolution -might be nonunitary in this case- of the target spin. The cost function is gate fidelity (and $n_0\cdot n_1$ is not usable in this method). 

In [33]:
from platform import python_version
print(python_version())
import sys 
sys.executable

3.8.5


'/Users/wenzheng/opt/anaconda3/bin/python'

In [2]:

def fidelity(U_i,U_r):
    """
    Fidelilty function
    """
    n = len(U_r)
    U_i = np.matrix(U_i)
    U_r = np.matrix(U_r)
    result = 1/(n+n*n)* (np.trace(U_r.getH() @ U_r) + abs(np.trace(U_i.getH() @ U_r))**2 )
    return (result)


## (1) Build the Hamiltonian for the system

In [3]:
Pauli_I = np.matrix([[1, 0],[ 0 , 1]])
Pauli_z = np.matrix([[1, 0],[ 0 ,-1]])
Pauli_x = np.matrix([[0, 1],[ 1 , 0]])

In [4]:
S_z =  np.matrix([[0, 0],[ 0 , -1]])

In [5]:
Pauli_I

matrix([[1, 0],
        [0, 1]])

In [6]:
"""
# The 3-n-spins in my NJP 
A_z_list = [2.0*np.pi* i for i  in [15.3/1000, 30.6/1000, 45.0/1000]]
A_x_list = [2.0*np.pi* i for i  in [12.9/1000, 25.7/1000, 20.0/1000]]
w_L= 2.0*np.pi*314/1000
"""

'\n# The 3-n-spins in my NJP \nA_z_list = [2.0*np.pi* i for i  in [15.3/1000, 30.6/1000, 45.0/1000]]\nA_x_list = [2.0*np.pi* i for i  in [12.9/1000, 25.7/1000, 20.0/1000]]\nw_L= 2.0*np.pi*314/1000\n'

In [7]:
# The three values Eve used?
A_z_list = [2.0*np.pi*i/1000 for i  in [15.3,17,20]]
A_x_list = [2.0*np.pi*i/1000 for i  in [12.9,17,20]]
w_L= 2.0*np.pi*314/1000

In [8]:


A_z_list = [2.0*np.pi*i/1000 for i  in [15.3,20]]
A_x_list = [2.0*np.pi*i/1000 for i  in [12.9,20]]
w_L= 2.0*np.pi*314/1000



In [9]:
A_z_list

[0.09613273519984768, 0.12566370614359174]

In [10]:
A_x_list 

[0.08105309046261666, 0.12566370614359174]

In [11]:
w_L

1.97292018645439

In [12]:
n_bath=len(A_z_list)

In [13]:
n_bath

2

$H=\frac{\omega_L}{2}Z+ S_z\otimes (A_x X +A_z Z) $

$ H = \frac{\omega_L}{2} I \otimes (Z \otimes I ... I + ... +I \otimes ... Z ) +\frac{S_z}{2} \otimes *\big(A_{x1} X \otimes ... \otimes I  + ... + I \otimes .... \otimes A_{xn}X\big)+ \frac{S_z}{2} \otimes *\big(A_{z1} Z \otimes ... \otimes I  + ... + I \otimes .... \otimes A_{zn}Z\big)
$ 

In [14]:
H_free = 0
for j in range(1,n_bath+1):
    H_free = H_free + 1/2* np.kron(Pauli_I,np.kron(np.identity(2**(j-1)), np.kron(w_L*Pauli_z, np.identity(2**(n_bath-j)))))


In [15]:
H_free;

In [16]:
H_sb = 0
for j in range(1,n_bath+1):
    H_sb = H_sb + 1/2* np.kron(S_z,np.kron(np.identity(2**(j-1)), np.kron(A_x_list[j-1]*Pauli_x, np.identity(2**(n_bath-j))))) \
                + 1/2* np.kron(S_z,np.kron(np.identity(2**(j-1)), np.kron(A_z_list[j-1]*Pauli_z, np.identity(2**(n_bath-j))))) 
H_sb;

In [17]:
for j in range(1,3+1):
    print(j)

1
2
3


In [18]:
H_total = H_free+H_sb

In [19]:
H_total ;

In [20]:
H_total_diag , T_total =  LA.eig( H_total ) # diagonlize the total Hamiltonian: H_diag = (T^-1).H.T
H_total_diag = np.diag(H_total_diag)

In [21]:
H_total_diag ;

In [22]:
inv(T_total) ;

In [23]:
np.linalg.norm(inv(T_total) @ T_total -np.identity(len(T_total)))   # T^-1 . T  = I  

1.5721033117243822e-16

In [24]:
np.allclose(inv(T_total) @ T_total , np.identity(len(T_total)))

True

In [25]:
np.allclose( inv(T_total)@H_total@T_total , H_total_diag )

True

In [26]:
np.linalg.norm(inv(T_total)@H_total@T_total - H_total_diag,'fro')

2.808415066818533e-15

consider the subsystem only includes NV + target n: 


In [27]:
H_subsystem = 1/2* np.kron(Pauli_I,w_L*Pauli_z) + \
    1/2*np.kron(S_z,A_x_list[0]*Pauli_x)+ 1/2*np.kron(S_z,A_z_list[0]*Pauli_z)

H_subsystem_diag, T_sub =  LA.eig( H_subsystem ) # diagonlize the total Hamiltonian: H_diag = (T^-1).H.T
H_subsystem_diag = np.diag(H_subsystem_diag)

In [28]:
#H_subsystem_diag-H_total_diag  // Works only for one n-spin
#H_subsystem-H_total // Works only for one n-spin

## (2) Evolution Operators

In [29]:
"""

t_list = np.linspace(0,10)
t_list;

"""

'\n\nt_list = np.linspace(0,10)\nt_list;\n\n'

#### (2.A) CPMG evolution

1. Build the evolution operator of CPMG
2. Calcualt the M function (Hanson PRL 2012) around $t_1$ ; and $P_x = (M+1)/2$

In [30]:
"""
Evolution operator of CPMG // Input: t (unit time), N

"""
U_diag_free =  lambda t: expm(-1j * H_total_diag *t/4 )

U_pi = np.kron( expm(-1j* np.pi /2.0 *Pauli_x),np.identity(2**n_bath))

U_cpmg_unit =  lambda t: (T_total@U_diag_free(t)@inv(T_total)) @ U_pi @ (T_total@U_diag_free(t)@U_diag_free(t) @inv(T_total)) @ U_pi @ (T_total@U_diag_free(t)@inv(T_total))   

U_cpmg_N = lambda t,N: matrix_power(U_cpmg_unit(t),N)


In [31]:
U_cpmg_N(3.2675,1) ;

We need to use U= $|0\rangle\langle 0| \otimes U_0+|1\rangle\langle 1| \otimes U_1 $ 

and we have $U_0 = \langle 0 | U |0 \rangle $, where $|0 \rangle = |0 \otimes I^{dim(bath)} \rangle$

Will use $U = V^N$ Since use the $M = Re Tr(V_0 U_1^{\dagger})$ function (Hanson PRL 2012)

In [32]:
proj_espin_0= np.kron(np.matrix([[1],[0]]), np.identity(2**n_bath))
proj_espin_1= np.kron(np.matrix([[0],[1]]), np.identity(2**n_bath))

U0_cpmg_extract =  lambda t,N: proj_espin_0.getH() @ U_cpmg_N(t,N) @  proj_espin_0 
U1_cpmg_extract =  lambda t,N: proj_espin_1.getH() @ U_cpmg_N(t,N) @  proj_espin_1
M_cpmg_func = lambda t,N: 1/(2**n_bath)* np.trace( U0_cpmg_extract(t,N) @ U1_cpmg_extract(t,N).getH() )
Px_cpmg_func = lambda t,N: (np.real(M_cpmg_func(t,N))+1)/2

In [33]:
# Let N =12 
[M_cpmg_func(i,12) for i in np.linspace(2,4,100)];

### (2.B) UDD evolution

Build the evolution operator under UDD-n 

In [34]:
"""
Evolution operator of UDD // Input: t (unit time),n, N

"""
def U_uddn_N(t,n,N,idle_spin ="on"):
    """
    Define the evolution operator for UDD-n // single unit
    if n is odd, the evo_op doubles, i.e. double units for odd
    In this way, the repetition N is odd/even integer 
    =================================================
    if idle_spin= "off", then only target spin
    """
    if idle_spin == "off":
        H_diag = H_subsystem_diag
        T = T_sub
        U_pi= np.kron( expm(-1j* np.pi /2.0 *Pauli_x),np.identity(2))
    else: 
        H_diag = H_total_diag
        H_diag = np.matrix(H_diag)
        T = T_total
        U_pi= np.kron( expm(-1j* np.pi /2.0 *Pauli_x),np.identity(2**n_bath))
    t_step = [ t*((np.sin(np.pi*j/(2*n+2)))**2-(np.sin(np.pi*(j-1)/(2*n+2)))**2) for j in range(1,n+2)]   # Good idea: use a small test to convince the range(1,n+2) is correct
    #print(t_step)     #print(np.shape(H_diag))     #print(np.shape(H_total_diag))     #print(T)
    U_diag_free_step = [expm(-1j * H_diag * i ) for i in t_step] # The free evolution parts between two pulses
    U_udd_result = 1
    for evol_piece in U_diag_free_step:
        U_udd_result = U_pi @ (T @ evol_piece @ inv(T)).dot(U_udd_result)
    U_udd_result = U_pi @ U_udd_result # Cancel the U_pi at the end of evolution
    if n%2 == 1:   # if n is odd, the evol should double
        U_udd_result = matrix_power(U_udd_result,2)
    U_udd_result = matrix_power(U_udd_result,N) 
    return(U_udd_result)

In [35]:
U_uddn_N(1,4,1,"on");
U_uddn_N(1,4,1,"off");

In [36]:
# Verify UDD-2 = CPMG case 
print(np.allclose(U_uddn_N(3.2,2,12) , U_cpmg_N(3.2,12)) )
print(np.allclose(U_uddn_N(1,2,24) , U_cpmg_N(1,24)) )
print(np.allclose(U_uddn_N(4,2,60) , U_cpmg_N(4,60)) )

True
True
True


In [37]:
proj_espin_0= np.kron(np.matrix([[1],[0]]), np.identity(2**n_bath))
proj_espin_1= np.kron(np.matrix([[0],[1]]), np.identity(2**n_bath))

U0_uddn_extract =  lambda t,n,N: proj_espin_0.getH() @ U_uddn_N(t,n,N) @  proj_espin_0 
U1_uddn_extract =  lambda t,n,N: proj_espin_1.getH() @ U_uddn_N(t,n,N) @  proj_espin_1
M_uddn_func = lambda t,n,N: 1/(2**n_bath)* np.trace( U0_uddn_extract(t,n,N) @ U1_uddn_extract(t,n,N).getH() )

Px_uddn_func = lambda t,n,N: (np.real(M_uddn_func(t,n,N))+1)/2

In [38]:
# Let N =48 for UDD-4
[Px_uddn_func(i,4,48) for i in np.linspace(2,4,100)];

## (3) Optimization & Fidelity 

============================================================

1. 1st spin as target spin

2. use t_1 and search/opt nearby

3. In the case of UDD4, the rotation axes are flipped compared to CPMG, thus we can set the flipped ideal gate for it. 
Ideal gates are $CR_{\pm} (\frac{\pi}{2})$ or  $CR_{\mp} (\frac{\pi}{2})$ 

============================================================

In [39]:
"""
Ideal total evolution operator -  CR(\pm \pi/2) 

"""
U_ideal= np.kron(np.kron(np.matrix([[1, 0],[ 0 , 0]]), expm(-1j*np.pi/4*Pauli_x)) \
 + np.kron(np.matrix([[0, 0],[ 0 , 1]]), expm(+1j*np.pi/4*Pauli_x)), np.identity(2**(n_bath-1)) )
"""
Another Ideal total evolution operator - opposite axes -  CR(\mp \pi/2) 

"""
U_ideal_mp= np.kron(np.kron(np.matrix([[1, 0],[ 0 , 0]]), expm(+1j*np.pi/4*Pauli_x)) \
 + np.kron(np.matrix([[0, 0],[ 0 , 1]]), expm(-1j*np.pi/4*Pauli_x)), np.identity(2**(n_bath-1)) )

In [41]:
U_ideal ;

#### (3.A) Test Pure CPMG

In [42]:
"""Build a resonance time range to search // around t1: (0.9~1.1)t1  """
t1_value = 4*np.pi/(2*w_L-A_z_list[0])
# set first n-spin as target 

t1_srch = np.linspace((1-0.1)*t1_value, (1+0.1)*t1_value, 300)
# Search data around t_1_value

In [43]:
cpmg_theta_t1 = 2* A_x_list[0]/(w_L-A_z_list[0]) # the rotation angle away from 2pi at t_1
cpmg_theta_t1

0.08637428858386342

In [44]:
np.pi/4/cpmg_theta_t1

9.092962457628595

Obtain the unoptimized N_udd is complicated (w/o analytical expression + )

In [45]:
"""
Build a N range to optimize
"""
# search range:  0.5~1.5 * N_{t_1}, step =1 
N_cpmg_t1_srch =np.int(0.5*np.pi/4/cpmg_theta_t1)+  np.arange(0, np.int(1.5*np.pi/4/cpmg_theta_t1)-np.int(0.5*np.pi/4/cpmg_theta_t1))
N_cpmg_t1_srch

array([ 4,  5,  6,  7,  8,  9, 10, 11, 12])

In [59]:
start_time = time.time()
###########################################
###########################################
t1_cpmg_opt=0
N_cpmg_opt =0
fid_cpmg_opt=0
for i in t1_srch:
    for j in N_cpmg_t1_srch:
        if abs(fidelity(U_ideal, U_cpmg_N(i,j)))>fid_cpmg_opt:
            fid_cpmg_opt=fidelity(U_ideal, U_cpmg_N(i,j))
            t1_cpmg_opt=i
            N_cpmg_opt=j
                     
print("t1_cpmg_opt = ", t1_cpmg_opt)
print("N_cpmg_opt = ", N_cpmg_opt)
print("fid_cpmg_opt = ", fid_cpmg_opt)
###########################################
###########################################
print("--- %s seconds ---" % (time.time() - start_time))

t1_cpmg_opt =  3.2762491586939335
N_cpmg_opt =  5
fid_cpmg_opt =  (0.6694727178070305-4.86153288195609e-20j)
--- 3.921926975250244 seconds ---


Running Time:
1. single-spin: 3s 
2. Two-spins: 5s 
3. Eight-spins:  40min

In [47]:
print(U_cpmg_N(3.27,12))
print(fidelity(U_ideal, U_cpmg_N(3.27,12)))

[[ 0.59456223+8.69743042e-02j -0.0574347 -6.25601314e-01j
   0.08014027-3.32075918e-01j -0.35715347+5.67374654e-15j
   0.        +0.00000000e+00j  0.        +0.00000000e+00j
   0.        +0.00000000e+00j  0.        +0.00000000e+00j]
 [-0.0574347 -6.25601314e-01j  0.56878713-1.93778143e-01j
  -0.35715347+2.50452639e-15j -0.08014027-3.32075918e-01j
   0.        +0.00000000e+00j  0.        +0.00000000e+00j
   0.        +0.00000000e+00j  0.        +0.00000000e+00j]
 [ 0.08014027-3.32075918e-01j -0.35715347+2.26327261e-15j
   0.56878713+1.93778143e-01j  0.0574347 -6.25601314e-01j
   0.        +0.00000000e+00j  0.        +0.00000000e+00j
   0.        +0.00000000e+00j  0.        +0.00000000e+00j]
 [-0.35715347+5.45838372e-15j -0.08014027-3.32075918e-01j
   0.0574347 -6.25601314e-01j  0.59456223-8.69743042e-02j
   0.        +0.00000000e+00j  0.        +0.00000000e+00j
   0.        +0.00000000e+00j  0.        +0.00000000e+00j]
 [ 0.        +0.00000000e+00j  0.        +0.00000000e+00j
   0.     

#### (3.B) Test Pure UDD

1. Try to avoid using odd n in UDD-n to make $t_1$ concide with CPMG.
2. Notice that due to the absence of $\phi_{udd}$ expression, it is impossible to get a value of $N$. 
3. To obtain the $\phi_{udd}$, we use the "subsystem" $U_{e,1} = |0\rangle \langle 0| \otimes R(\theta)_1+|1\rangle \langle 1|\otimes R(-\theta)_1$ (Note that the opposite $\theta$ value is accurate for CPMG and UDD-3); then we extract value by using $\langle 0|U_{e,1}|0\rangle$, whose trace is $2\cos\theta$.


In [60]:
"""Build a resonance time range to search // around t1:  (0.9~1.1)t1 """
t1_uddn_value = 4*np.pi/(2*w_L-A_z_list[0])
print(t1_uddn_value)
# set first n-spin as target 

t1_uddn_srch = np.linspace((1-0.1)*t1_value, (1+0.1)*t1_value, 100)
# Search data around t_1_value

3.264240248082259


In [61]:
"""
Build a N range to optimize
We borrow the result from CPMG: Let UDD-n (even) range fall into 
[Ncp,10*Ncp]
"""
# search range:  1~10 * N_cp_{t_1}, step =1 
#N_uddn_t1_srch =np.int(np.pi/4/cpmg_theta_t1)+  np.arange(0, np.int(10*np.pi/4/cpmg_theta_t1))
#N_uddn_t1_srch

'\nBuild a N range to optimize\nWe borrow the result from CPMG: Let UDD-n (even) range fall into \n[Ncp,10*Ncp]\n'

In [62]:
"""
Build a N range to optimize
This time, to narrow down searching range, we need to 
numerical find the N_udd_t1 by calculating theta_t1_udd of the TARGET spin, 
where t1 is the analytical  reson_time (NOT optimized). 
"""
proj_sub_e_0= np.kron(np.matrix([[1],[0]]), np.identity(2)) # 

theta_uddn_t1_value =  np.arccos(1/2*np.trace(proj_sub_e_0.getH()@ U_uddn_N(t1_uddn_value,4,1,"off") @ proj_sub_e_0))
print(theta_uddn_t1_value)
N_uddn_t1_value = np.int( 1/4*np.pi/theta_uddn_t1_value.real )
print(N_uddn_t1_value)
N_uddn_t1_srch = N_uddn_t1_value-10 + np.arange(0,20)
N_uddn_t1_srch

(0.01131728244748097+7.357641265839775e-15j)
69


array([59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75,
       76, 77, 78])

In [63]:
start_time = time.time()
###########################################
###########################################
"""
Optimize UDD-4

"""
t1_uddn_opt=0
N_uddn_opt =0
fid_uddn_opt=0
for i in t1_uddn_srch:
    for j in N_uddn_t1_srch:
        if abs(fidelity(U_ideal_mp, U_uddn_N(i,4,j)))>fid_uddn_opt:
            fid_uddn_opt=fidelity(U_ideal_mp, U_uddn_N(i,4,j))
            t1_uddn_opt=i
            N_uddn_opt=j
print("UDD-4")            
print("t1_uddn_opt = ", t1_uddn_opt)
print("N_uddn_opt = ", N_uddn_opt)
print("fid_uddn_opt = ", fid_uddn_opt)
###########################################
###########################################
print("--- %s seconds ---" % (time.time() - start_time))

UDD-4
t1_uddn_opt =  3.2543486109668587
N_uddn_opt =  78
fid_uddn_opt =  (0.6365924297906616+4.410042092046956e-19j)
--- 4.3655149936676025 seconds ---


Running Time:
1. Single spin: 4s
2. Eight-spins: 

In [64]:
U_uddn_N(3.261,4,78)
print("fidelity= ",fidelity(U_ideal_mp,U_uddn_N(3.261,4,78)))

fidelity=  (0.6079465791038206-1.1220814551809946e-20j)


### (3.C) Test hybrid : CPMG +UDD-4

Notice that to simplify the calculation, we find used the 
$t_{cpmg}$ and $N_{cpmg}$ that are optimized from CPMG case alone;
then we $\textbf{optimize}$ the $t_{udd}$ and $N_{udd}$.

Otherwise, we need to optmize four paratmeters simultaneously.

Another important thing to remember is since CPMG produce $CR(\pm \theta)$ and UDD4 produces $CR(\mp \theta)$, their rotation angles are oppostite, so in the hybrid situation, we let the CPMG over rotate by using
$$\begin{equation}
N_{cpmg} := N_{cpmg}+1.
\end{equation}$$
So we alwasy do one more for the optimized $N_{cpmg}$ and then optimizing $N_{udd}$ to counteract the over-rotation and let total rotation good. 

In [57]:
print(t1_cpmg_opt)
print(N_cpmg_opt)

3.2762491586939335
5


In [58]:
print(t1_uddn_opt)
print(N_uddn_opt)

3.2543486109668587
78


In [92]:
"""Build a resonance time range to search // around t1"""
t1_value = 4*np.pi/(2*w_L-A_z_list[0])
# set first n-spin as target 

t1_hybrid_srch = np.linspace((1-0.1)*t1_value, (1+0.1)*t1_value, 300)
# Search data around t_1_value

In [93]:
"""
Build a N range to optimize
Since the UDD part is a small appending part we set the 0< N_udd < 0.5* N_cpmg 
"""
# search range:  0 ~0.5*N_cp_{t_1}, step =1 

N_hybrid_t1_srch = np.arange(0, np.int(0.5*np.pi/4/cpmg_theta_t1))
N_hybrid_t1_srch

array([0, 1, 2, 3])

In [94]:
start_time = time.time()
###########################################
###########################################
"""
Optimize UDD-4 in hybrid protocol

"""
t1_hybrid_opt=0
N_hybrid_opt =0
fid_hybrid_opt=0
U_cpmg_in_hybrid = U_cpmg_N(t1_cpmg_opt,N_cpmg_opt+1) ## N_cpmg := N_cpmg+1 over-rotate
for i in t1_hybrid_srch:
    for j in N_hybrid_t1_srch:
        if abs(fidelity(U_ideal, U_uddn_N(i,4,j)@ U_cpmg_in_hybrid))>fid_hybrid_opt:
            fid_hybrid_opt=fidelity(U_ideal, U_uddn_N(i,4,j)@ U_cpmg_in_hybrid)
            t1_hybrid_opt=i
            N_hybrid_opt=j
print("UDD-4 in hybrid")            
print("t1_hybrid_opt = ", t1_hybrid_opt)
print("N_hybrid_opt = ", N_hybrid_opt)
print("fid_hybrid_opt = ", fid_hybrid_opt)
###########################################
###########################################
print("--- %s seconds ---" % (time.time() - start_time))

UDD-4 in hybrid
t1_hybrid_opt =  2.9378162232740332
N_hybrid_opt =  0
fid_hybrid_opt =  (0.9600903899782772+1.3986717796678656e-19j)
--- 2.2518558502197266 seconds ---
